In [ ]:
import pandas as pd
import io, numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# --- Data -------------------------------------------------
raw_table = """
Type\tCitation\tParam\tCategory
General\t2024\t14092815\tfeature
Medical (locked)\t2024\t34523650\tDeformable
Medical\t2024\t8897107\tDeformable through Spatial Landmark Detection
Medical\t2024\t8922444\tdeformable, affine
General\t2023\t303424\tfeature
Medical\t2024\t38449920\tDeformable Non-Rigid Registration Volumetric; Coarse-to-Fine
Medical/Histological\t2024\t450564\tdeformable
Medical (locked)\t2024\t1491\tdeformable
General/Medical\t2024\t1076964\tdeformable
Medical\t2024\t7310000\tMulti-modal deformable registration
Medical\t2024\t53007765\tdeformable
Medical\t2024\t330000\tDeformable IR
Medical\t2024\t896551\tDeformable IR
Medical\t2024\t328791\tAffine IR
Medical\t2024\t615076\tDeformable and Rigid IR
Medical\t2024\t9640000\tMulti-modal deformable registration
Medical\t2024\t163737623\tDeformable IR
Remote Sensing\t2024\t7503968\tCoarse-Fine IR
Medical\t2024\t133379\tdeformable IR
Remote Sensing\t2024\t7216893\tCoarse-Fine IR
Remote Sensing\t2024\t31096129\tdeformable
Remote Sensing\t2024\t4902300\tFeature based IR, Keypoint Matching
General/Medical\t2023\t960998\tCoarse-Fine IR
General/Medical\t2023\t165659\trigid and deformable
Medical\t2023\t112350291\tdeformable IR
General\t2023\t8630497\tDeformable/affine IR
General\t2023\t67319427\tfeature
Medical\t2023\t133379\tdeformable IR
Medical\t2023\t18290716\tdeformable IR
Medical\t2023\t233476\tdiffeomorphic deformable IR
Remote Sensing\t2023\t7549640\tfeature
Medical\t2024\t10180163\tDeformable IR
Medical\t2023\t3222\tAffine IR
Medical\t2023\t133379\tDeformable IR
Medical\t2023\t16318691\tDeformable IR
Medical\t2023\t133379\tDeformable IR
Medical\t2023\t53010687\tdeformable
Medical\t2023\t13140670\tdeformable
Medical\t2023\t5706508\tCoarse-Fine IR
General\t2023\t77430931\tdeformable IR
Medical\t2023\t1029670\tDeformable IR
Medical\t2023\t526448\tDeformable IR, multimodal volumetric
Medical\t2023\t4085458\tfeature
Remote Sensing\t2023\t7151276\tdeformable
General\t2022\t1962429\tcross modal deformable IR
Medical/General/Remote Sensing\t2023\t960998\tdeformable affine IR
Medical\t2022\t12030208\tAffine Registration
Medical\t2022\t133379\tDeformable IR
Medical\t2022\t7976611\tDeformable Registration
Medical/General\t2022\t90672387\tDeformable Registration
Medical\t2022\t11721891\tDeformable
Medical\t2022\t21002443\tDeformable
Medical\t2022\t16049667\tDeformable
Medical\t2022\t327331\tDeformable
Medical\t2022\t1029670\tDeformable
Medical\t2022\t45481080\tDeformable
Medical\t2022\t18290716\tDeformable
Remote Sensing\t2022\t13841096\tFeature-based
Medical\t2022\t5827427\tDeformable
Medical\t2022\t336080\tdiffeomorphic
Medical\t2022\t1248777\tDeformable, Feature-based
Medical\t2022\t608112\tDeformable
Medical\t2022\t108611336\tDeformable
Remote Sensing\t2022\t13600229\tDeformable
Medical\t2021\t2374072\tDeformable, Groupwise
Medical\t2021\t45510153\tDeformable
Medical\t2021\t722598\tDeformable Image Registration
Medical\t2021\t31560079\tDeformable Image Registration
Medical\t2021\t608112\tDeformable Image Registration
Medical\t2021\t23007954\tDeformable
Remote Sensing\t2021\t40847185\tfeature
Medical\t2021\t491668\tDeformable
Medical\t2021\t1739270\tDeformable
Medical\t2021\t11774539\tDeformable
Remote Sensing\t2021\t95266\tDeformable
"""
df = pd.read_csv(io.StringIO(raw_table), sep="\t")

# --- classification functions ----------
def get_type_group(t: str) -> str:
    t_low = t.lower()
    if "remote sensing" in t_low:
        return "Remote Sensing"
    elif "medical" in t_low:
        return "Medical"
    else:
        return "Other"

def get_broad_cat(c: str) -> str:
    c_low = c.lower()
    if "feature" in c_low:
        return "Feature‑based"
    elif "deformable" in c_low:
        return "Deformable"
    else:
        return "Other"

df["TypeGroup"] = df["Type"].apply(get_type_group)
df["BroadCat"] = df["Category"].apply(get_broad_cat)
df["Param"] = pd.to_numeric(df["Param"], errors="coerce")
df.dropna(subset=["Param"], inplace=True)

# keep only two domains
df = df[df["TypeGroup"].isin(["Medical", "Remote Sensing"])]
df["LogParam"] = np.log10(df["Param"])

years = sorted(df["Citation"].unique())  # 2021-2024
cats  = ["Feature‑based", "Deformable", "Other"]
types = ["Medical", "Remote Sensing"]

data_for_boxes = []
positions      = []
pos_map        = {}
pos_counter = 0
group_width = len(cats)*len(types)
gap = 0.5  # squeeze x‑axis

for year in years:
    for cat_idx, cat in enumerate(cats):
        for typ_idx, typ in enumerate(types):
            subset = df[(df["Citation"]==year) & (df["BroadCat"]==cat) & (df["TypeGroup"]==typ)]
            if not subset.empty:
                data_for_boxes.append(subset["LogParam"].values)
                current_pos = pos_counter + cat_idx*len(types)+typ_idx
                positions.append(current_pos)
                pos_map[(year, cat, typ)] = current_pos
    pos_counter += group_width + gap

# --- plotting ----------
plt.rcParams.update({'font.size': 14})
fig, ax = plt.subplots(figsize=(10,6))

box = ax.boxplot(
    data_for_boxes,
    positions=positions,
    widths=0.4,
    patch_artist=True,
    showfliers=False
)

# colorize boxes by BroadCat
cat_colors = {"Feature‑based":"#4e79a7", "Deformable":"#f28e2b", "Other":"#59a14f"}
for b, pos in zip(box["boxes"], positions):
    # need to figure cat of that pos
    # reverse map
    cat_of_box = None
    for (yr, cat, typ), p in pos_map.items():
        if p == pos:
            cat_of_box = cat
            break
    b.set_facecolor(cat_colors.get(cat_of_box,"#bab0ac"))
    b.set_alpha(0.55)
    b.set_edgecolor('black')

# whiskers color
for whisk in box['whiskers']:
    whisk.set_color('black')
for cap in box['caps']:
    cap.set_color('black')
for median in box['medians']:
    median.set_color('black')

# scatter overlay
for _, row in df.iterrows():
    combo = (row["Citation"], row["BroadCat"], row["TypeGroup"])
    if combo in pos_map:
        jitter = (np.random.rand()-0.5)*0.3
        marker = 'o' if row["TypeGroup"]=="Medical" else '^'
        ax.scatter(pos_map[combo]+jitter, row["LogParam"],
                   marker=marker, s=40, edgecolors='k', linewidths=0.4, alpha=0.7)

# x ticks
centre_points = []
labels   = []
pos_counter = 0
for y in years:
    centre_points.append(pos_counter + (group_width-1)/2)
    labels.append(str(y))
    pos_counter += group_width + gap
ax.set_xticks(centre_points)
ax.set_xticklabels(labels)

ax.set_ylabel("log\u2081\u2080(Parameter count)")
# ax.set_title("Image‑registration models: parameter distribution by year/domain", fontsize=18)

# legends
legend_elems = [
    Line2D([0],[0], marker='o', linestyle='', label='Medical', markeredgecolor='k',markerfacecolor='white'),
    Line2D([0],[0], marker='^', linestyle='', label='Remote Sensing', markeredgecolor='k',markerfacecolor='white'),
    Line2D([0],[0], color=cat_colors["Feature‑based"], lw=10, label='Feature‑based IR'),
    Line2D([0],[0], color=cat_colors["Deformable"], lw=10, label='Deformable IR'),
    Line2D([0],[0], color=cat_colors["Other"], lw=10, label='Other IR')
]
ax.legend(handles=legend_elems, loc='upper left', bbox_to_anchor=(1,1))
fig.tight_layout()


In [ ]:
# -----------------------------------------------------------
#  0. Imports & set a minimal, clean Matplotlib style
# -----------------------------------------------------------
import re, unicodedata, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.autolayout": True,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    # "axes.prop_cycle": plt.cycler()        # let Matplotlib choose pleasant default colours
})

# -----------------------------------------------------------
#  1. Load the raw table you provided
# -----------------------------------------------------------
df = pd.read_csv("/Users/sayanmandal/workspaces/tests/cleaned.csv", dtype=str).fillna("")   # keep everything as text for safe cleaning

# -----------------------------------------------------------
#  2. Normalisation helpers
# -----------------------------------------------------------
# 2‑A. Unicode “defuzzer” – swap weird spaces/dashes for ASCII
def _defuzz(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)             # standard unicode form
    return (text
            .replace("‑", "-")           # non‑breaking hyphen
            .replace(" ", " ")           # narrow no‑break space
            .replace(" ", " ")           # non‑break space
            .strip())

# 2‑B. Canonical replacements (typos, variants → golden form)
CANONICAL = {
    r"\bmutliview\b":      "multiview",
    r"\bmutlitemporal\b":  "multitemporal",
    r"\bmutlispectral\b":  "multispectral",
    r"\bRemotesensing\b":  "Remote Sensing",
    r"\bmedical\b":        "Medical",        # lower‑case variants
    r"\bselfsupervised\b": "semisupervised", # fold into ‘semi’ bucket
    r"\bweaklysupervised\b":"semisupervised",
    r"\bunsupervised,?semisupervised\b": "semisupervised",
    r"\bclassical,?unsupervised\b": "classical",  # keep classical as its own
}

def _canonise(text: str) -> str:
    out = text
    for pat, repl in CANONICAL.items():
        out = re.sub(pat, repl, out, flags=re.I)
    return out

# 2‑C. Split a multi‑value cell into a *sorted*, de‑duplicated list
_SPLIT_RE = re.compile(r"[,+;/]| and |&|\s*\+\s*")
def _tokenise(cell: str) -> list[str]:
    toks = [tok.strip() for tok in _SPLIT_RE.split(cell) if tok.strip() and tok.strip().lower() != "<blank>"]
    # kill empty / “<blank>”
    return sorted(dict.fromkeys(toks))           # de‑dup & stable order

# -----------------------------------------------------------
#  3. Apply all cleaners column‑wise
# -----------------------------------------------------------
for col in df.columns:
    df[col] = (df[col]
                 .astype(str)
                 .map(_defuzz)
                 .map(_canonise))

# values that are literally '<blank>' → ""
df.replace(r"^\s*<blank>\s*$", "", regex=True, inplace=True)

df = df[['Year', 'Type', 'Category', 'Acquisition', 'Network', 'Objective']]

# -----------------------------------------------------------
#  4. Save a **clean copy** – same 6‑column, comma‑delimited CSV
# -----------------------------------------------------------
df.to_csv("papers_cleaned.csv", index=False)

# -----------------------------------------------------------
#  5. Helper – explode multivalue columns so each token gets its own row
# -----------------------------------------------------------
def explode_on(df_in: pd.DataFrame, column: str) -> pd.DataFrame:
    """Return df with `column` exploded into one token per row (lower‑cased for grouping)"""
    out = df_in.copy()
    out[column] = out[column].map(_tokenise)
    return out.explode(column, ignore_index=True).rename(columns={column: column.lower().strip()})

# -----------------------------------------------------------
#  6. Plotter – one bar chart per facet
# -----------------------------------------------------------
def plot_yearwise_counts(data: pd.DataFrame, facet: str, ax=None):
    facet_clean = facet.replace(" ", "_").lower()
    exploded = explode_on(data, facet)
    counts   = (exploded
                .groupby(["Year", facet_clean])
                .size()
                .unstack(fill_value=0)
                .sort_index())                      # index=Year

    # order columns by total frequency (helps legend readability)
    counts = counts.loc[:, counts.sum().sort_values(ascending=False).index]

    counts.plot(kind="bar",
                stacked=False,
                width=0.8,
                ax=ax,
                legend=True)

    ax.set_title(f"Year‑wise distribution of {facet}")
    ax.set_ylabel("Number of papers")
    ax.set_xlabel("Year")
    ax.set_xticklabels(counts.index.astype(str), rotation=0)

# -----------------------------------------------------------
#  7. Generate the five requested charts
# -----------------------------------------------------------
FIGSIZE = (12, 6)
facets  = ["Type", "Category", "Acquisition", "Network", "Objective"]

for facet in facets:
    fig, ax = plt.subplots(figsize=FIGSIZE)
    plot_yearwise_counts(df, facet, ax=ax)
    plt.show()

# -----------------------------------------------------------
#  8. (Optional) Total papers surveyed per year
# -----------------------------------------------------------
totals = df.groupby("Year").size()
print("\nTotal papers surveyed per year:\n", totals)


In [ ]:
splitter = re.compile(r"[,+;/]| and |&|\s*\+\s*")
# df["Type"] = df["Type"].apply(lambda x: [t.strip() for t in splitter.split(x) if t.strip()])

exploded = df.explode("Type", ignore_index=True)
counts   = exploded.groupby(["Year", "Type"]).size().unstack(fill_value=0)

ax = counts.plot(kind="bar", stacked=True, figsize=(10, 5), width=0.3)
plt.xlabel("Year")
plt.ylabel("Number of papers")

# ‑‑‑ requested tweaks ‑‑‑
plt.legend(title="Domain", loc="upper right", bbox_to_anchor=(1.3, 1.0))   # move legend
plt.xticks(rotation=45, ha="right")           # rotate x‑tick labels

plt.savefig("domain_dist.png", bbox_inches="tight", dpi=300)

In [ ]:
df = pd.read_csv("papers_cleaned.csv", dtype=str).fillna("")   # keep everything as text for safe cleaning

In [ ]:
splitter = re.compile(r"[,+;/]| and |&|\s*\+\s*")
# df["Category"] = df["Category"].apply(lambda x: [c.strip() for c in splitter.split(x) if c.strip() and c.strip().lower() != "multi-modal"])
# df['Category'] = df['Category'].apply(lambda x: [c for c in x if c.lower() != "multi-modal"])
# 3) Explode so each (Year, Category) is its own row
exploded = df.explode("Category", ignore_index=True)

# 4) Year × Category frequency table
counts = exploded.groupby(["Year", "Category"]).size().unstack(fill_value=0)

# 5) Plot – stacked bars, legend top‑right, x‑ticks 45°
ax = counts.plot(kind="bar", stacked=True, figsize=(10, 5), width=0.3)
plt.xlabel("Year")
plt.ylabel("Number of papers")
plt.legend(title="Category", loc="upper right", bbox_to_anchor=(1.3, 1.0))
plt.xticks(rotation=45, ha="right")
plt.savefig("category_dist.png", bbox_inches="tight", dpi=300)

In [ ]:
splitter = re.compile(r"[,+;/]| and |&|\s*\+\s*")
# df["Network"] = df["Network"].apply(lambda x: [n.strip().title() for n in splitter.split(x) if n.strip()])

# 3) Explode so each (Year, Network) has its own row
exploded = df.explode("Network", ignore_index=True)

# 4) Build Year × Network frequency table
counts = exploded.groupby(["Year", "Network"]).size().unstack(fill_value=0)

# 5) Plot – stacked bars
ax = counts.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.xlabel("Year")
plt.ylabel("Number of papers")

# requested tweaks
# convert legends to title case
counts.columns = [col.title() for col in counts.columns]
plt.legend(title="Network", loc="upper right", bbox_to_anchor=(1.3, 1.0))
plt.xticks(rotation=45, ha="right")
plt.savefig("network_dist.png", bbox_inches="tight", dpi=300)

In [ ]:
splitter = re.compile(r"[,+;/]| and |&|\s*\+\s*")
df["Acquisition"] = df["Acquisition"].apply(lambda x: [a.strip().title() for a in splitter.split(x) if a.strip()])

# 3) Explode so each (Year, Acquisition) pair is its own row
exploded = df.explode("Acquisition", ignore_index=True)

# 4) Build a Year × Acquisition frequency table
counts = exploded.groupby(["Year", "Acquisition"]).size().unstack(fill_value=0)

# 5) Plot – stacked bars, legend top‑right, x‑ticks rotated 45°
ax = counts.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.xlabel("Year")
plt.ylabel("Number of papers")
plt.legend(title="Acquisition Methods", loc="upper right", bbox_to_anchor=(1.3, 1.0))
plt.xticks(rotation=45, ha="right")
plt.savefig("acquisition_dist.png", bbox_inches="tight", dpi=300)

In [ ]:
splitter = re.compile(r"[,+;/]| and |&|\s*\+\s*")
# df["Objective"] = df["Objective"].apply(lambda x: [o.strip().title() for o in splitter.split(x) if o.strip()])

# 3) Explode so each (Year, Objective) is its own row
exploded = df.explode("Objective", ignore_index=True)

# 4) Year × Objective frequency table
counts = exploded.groupby(["Year", "Objective"]).size().unstack(fill_value=0)

# 5) Plot – **grouped** bars (stacked=False)
ax = counts.plot(kind="bar", stacked=False, figsize=(12, 6), width=0.8)
plt.xlabel("Year")
plt.ylabel("Number of papers")
plt.legend(title="Objective", loc="upper right", bbox_to_anchor=(1.3, 1.0), fontsize='small', prop={'size': 10})
plt.xticks(rotation=45, ha="right")
plt.savefig("objective_dist.png", bbox_inches="tight", dpi=300)